# Task 1 — Text Pre-processing

## Table of Contents

1. [Import Libraries](#1-import-lib)
2. [Helper Functions](#2-helper-functions)
   - 2.1 [Constants & Paths](#constants--paths)
   - 2.2 [Core Functions](#core-functions)
3. [Main Logic](#3-main-logic)
   - 3.1 [Examining and Loading Data](#31-examining-and-loading-data)
   - 3.2 [Pre-processing Data](#32-pre-processing-data)
4. [Appendix — Stopword Investigation](#4-appendix)
   - 4.1 [ClaudeTextExpert Class](#claudetextexpert-class)
   - 4.2 [Stopword Utilities](#stopword-utilities)
   - 4.3 [Running the Audit](#running-the-audit)

---

## Installation

> **Requires Python 3.10+**

```bash
pip install pandas nltk pyspellchecker rapidfuzz wordfreq tqdm python-dotenv anthropic
```

Or if you are using `uv` / `pyproject.toml`:

```bash
uv pip install pandas nltk pyspellchecker rapidfuzz wordfreq tqdm python-dotenv anthropic
```

After installing, download the required NLTK data (run once):

```python
import nltk
nltk.download("wordnet")
nltk.download("omw-1.4")
```

# 1. Import Lib

In [13]:
import os
import re
import pandas as pd
from collections import Counter
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
import nltk
import re
from spellchecker import SpellChecker
from rapidfuzz import fuzz
from wordfreq import word_frequency
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
from math import ceil
from tqdm.auto import tqdm
import textwrap

# 2. Helper Functions

## Constants & Paths

In [2]:
# Constants
TOKENIZER_PATTERN = re.compile(r"[a-zA-Z]+(?:[-'][a-zA-Z]+)?") # From documentation


# Input:
# Path to the data file from your local
DATA_PATH = f"./data/cosmetics_beauty_products_reviews.csv"
STOPWORDS_PATH = f"./data/stopwords_en.txt"
ENHANCED_PATH = f"./data/stopwords_en_enhancement.txt" 

# Output path:
BASE_PATH = "./data"
VOCAB_PATH = f"{BASE_PATH}/vocab.txt"
OUTPUT_PATH = f"{BASE_PATH}/processed.csv"

## Core Functions

In [3]:
def load_stopwords(path: str) -> list[str]:
    """Load stopwords from a text file.

    Args:
        path: Path to a stopword file with one word per line.

    Returns:
        List of lowercase stopwords with empty lines removed.
    """
    # Read each non-empty line, strip whitespace and lowercase for consistent lookup
    with open(path, "r", encoding="utf-8") as f:
        words: list[str] = [line.strip().lower() for line in f if line.strip()]
    return words


def save_vocab(processed_docs: list[list[str]], path: str) -> dict:
    """
    Build a vocabulary from a list of tokenized documents.

    Args:
        processed_docs: Tokenized documents after preprocessing.
        path: Output file path for vocabulary entries.

    Returns:
        Dict-like vocabulary mapping represented in the saved file as word:index.
    """
    # Flatten all token lists into a set to deduplicate across all documents
    vocab_set = {t for doc in processed_docs for t in doc}
    # Sort alphabetically so the index assignment is deterministic across runs
    vocab_sorted = sorted(vocab_set)  # A–Z (ASCII-ish; digits would sort before letters if any)
    lines = []
    for i, word in enumerate(vocab_sorted):
        # Format: "word:index" — one entry per line
        lines.append(f"{word}:{i}")

    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines) + "\n")
    print(f"\n[Saved] {len(lines)} words → {path}")


def _split_tokens(text: str) -> list[str]:
    """Split raw text into alphabetic tokens.

    Args:
        text: Input text to tokenize.

    Returns:
        List of regex-matched tokens.
    """
    # Apply the compiled regex to extract alphabetic tokens (allows internal hyphens/apostrophes)
    return TOKENIZER_PATTERN.findall(str(text))


def _process_token(t: str, stop_set: set[str]) -> str | None:
    """Apply basic token filtering rules.

    Args:
        t: Candidate token string.
        stop_set: Set of stopwords for fast lookup.

    Returns:
        Normalized token if valid, otherwise None.
    """
    # Reject None / empty strings produced by upstream steps
    if t:
        # Single-character tokens carry no meaning — discard them
        if len(t) >= 2:
            # Keep the token only if it is not a stopword
            if t not in stop_set:
                return t
            else:
                return None
        else:
            return None
    else:
        return None
    

def _is_meaningful(word: str) -> bool:
    """Heuristic check for short-token quality.

    Args:
        word: Candidate token.

    Returns:
        True when token should be kept, else False.
    """
    # 2-char words must be genuinely common (by/as/in pass, ab/ac fail)
    if len(word) == 2 and word_frequency(word, 'en') < 1e-4:
        return False
    return True


def check_and_fix(lemmatizer: WordNetLemmatizer, speller: SpellChecker, word: str) -> str | None:
    """Spell-correct and lemmatize a token when possible.

    Args:
        lemmatizer: NLTK lemmatizer instance.
        speller: SpellChecker instance for candidate correction.
        word: Token to validate and normalize.

    Returns:
        Cleaned lemma token, or None when token should be discarded.
    """
    # Nothing to process for empty/None tokens
    if not word:
        return None

    # Collapse repeated characters: "goood" → "god" (hard) / "good" (light)
    # c_light: only 3+ repeats collapsed to 1;  c_hard: any repeat collapsed to 1
    c_light = re.sub(r'(.)\1{2,}', r'\1', word)
    c_hard  = re.sub(r'(.)\1+',    r'\1', word)

    if speller.known([word]):
        # Word is in the dictionary — drop it if it is too rare / meaningless
        if not _is_meaningful(word):
            return None
        # Accept the spell-checker's suggestion only when it is close enough (≥70 similarity)
        correction = speller.correction(word)
        if correction and _is_meaningful(correction) and fuzz.ratio(word, correction) >= 70:
            return lemmatizer.lemmatize(correction, pos=wordnet.NOUN)
        # Word is already correct — just lemmatize it
        return lemmatizer.lemmatize(word, pos=wordnet.NOUN)

    # Word is unknown — try progressively de-duplicated forms (hard → light → original)
    # dict.fromkeys preserves order and removes duplicates in case forms coincide
    for candidate in dict.fromkeys([c_hard, c_light, word]):
        if not candidate or not _is_meaningful(candidate):
            continue
        correction = speller.correction(candidate)
        if not correction or not _is_meaningful(correction):
            continue
        # Accept correction only when it is visually similar to the candidate
        if fuzz.ratio(candidate, correction) >= 70:
            return lemmatizer.lemmatize(correction, pos=wordnet.NOUN)

    # No valid form found — discard the token
    return None


def _filter_by_frequency(docs: list[list[str]], top_k_df: int = 20, min_tf: int = 2) -> list[list[str]]:
    """Remove low-value tokens using corpus frequencies.

    Args:
        docs: List of tokenized documents.
        top_k_df: Number of most common document-frequency terms to drop.
        min_tf: Minimum corpus term frequency threshold.

    Returns:
        Filtered tokenized documents.
    """
    # Count global term frequency (TF) and document frequency (DF) across the corpus
    tf: Counter[str] = Counter()
    df: Counter[str] = Counter()
    for doc in docs:
        tf.update(doc)
        df.update(set(doc))  # set() ensures each word counted once per document

    # Remove tokens that appear fewer than min_tf times in the whole corpus
    docs = [[t for t in doc if tf[t] >= min_tf] for doc in docs]

    # Remove the top_k_df most frequent words — they appear in nearly every doc and carry little signal
    top_df_words: set[str] = {w for w, _ in df.most_common(top_k_df)}
    docs = [
        # If filtering removes all tokens, keep the original to avoid empty documents
        filtered if (filtered := [t for t in doc if t not in top_df_words]) else doc
        for doc in docs
    ]
    return docs


def _setup():
    """Prepare required resources and output directories.

    Returns:
        None.
    """
    # Download required NLTK data files if not already present (wordnet for lemmatization)
    print("Installing nltk packages...")
    nltk.download("wordnet", quiet=True)
    nltk.download("omw-1.4", quiet=True)

    # Ensure the output directory exists before any file writes
    print(f"Creating directory: {BASE_PATH}")
    os.makedirs(os.path.dirname(BASE_PATH), exist_ok=True)
    

def _preprocess_chunk(
    chunk: list[str],
    stop_set: set[str],
    worker_id: int,
) -> list[list[str]]:
    """Preprocess one text chunk inside a worker thread.

    Args:
        chunk: Subset of raw review texts.
        stop_set: Stopword set shared across workers.
        worker_id: Worker index for progress display.

    Returns:
        Tokenized documents for this chunk.
    """
    # Each worker owns its own lemmatizer and spell-checker — not thread-safe to share
    lemmatizer = WordNetLemmatizer()
    speller = SpellChecker()
    docs = []
    for text in tqdm(chunk, desc=f"Core {worker_id + 1}", position=worker_id, leave=True):
        tokens = []
        for token in _split_tokens(text):
            t = token.lower()
            # First pass: discard single-char tokens and stopwords
            t = _process_token(t, stop_set)
            # Spell-check, de-duplicate characters, and lemmatize
            t = check_and_fix(lemmatizer, speller, t)
            # Second pass: lemmatization may produce a stopword (e.g. "be") — filter again
            t = _process_token(t, stop_set)
            if t:
                tokens.append(t)
        docs.append(tokens)
    return docs
    

def preprocess(
    texts: list[str],
    stopwords: list[str],
    top_k_df: int = 20,
    min_tf: int = 2,
    n_jobs: int = -1,  # -1 = all available cores
) -> list[list[str]]:
    """Run full text preprocessing with optional multithreading.

    Args:
        texts: Raw text documents.
        stopwords: Stopword list for token filtering.
        top_k_df: Number of highest document-frequency tokens to remove.
        min_tf: Minimum token frequency threshold across corpus.
        n_jobs: Worker count (-1 uses all CPU cores).

    Returns:
        Preprocessed tokenized documents.
    """
    # Build a set for O(1) stopword lookup; normalise case/whitespace upfront
    stop_set: set = set(w.lower().strip() for w in stopwords if w.strip())

    # Resolve worker count: cap at available CPUs; always at least 1
    n_workers: int = os.cpu_count() if n_jobs == -1 else min(n_jobs, os.cpu_count())
    n_workers = max(1, n_workers)

    # Split texts into equal-sized chunks — one chunk per worker thread
    chunk_size: int = ceil(len(texts) / n_workers)
    chunks: list[list[str]] = [texts[i : i + chunk_size] for i in range(0, len(texts), chunk_size)]
    actual_workers: int = len(chunks)  # may be < n_workers for tiny inputs

    # Pre-allocate result list so we can insert by index regardless of completion order
    ordered: list[list[list[str]]] = [None] * actual_workers
    with ThreadPoolExecutor(max_workers=actual_workers) as executor:
        futures: dict = {
            executor.submit(_preprocess_chunk, chunk, stop_set, i): i
            for i, chunk in enumerate(chunks)
        }
        # Collect results as they finish; use the stored index to preserve original order
        for future in as_completed(futures):
            idx = futures[future]
            ordered[idx] = future.result()

    # Flatten the list-of-chunks back into a flat list of documents
    docs: list[list[str]] = [doc for chunk_docs in ordered for doc in chunk_docs]

    # Apply corpus-level frequency filtering once across all workers' output.
    return _filter_by_frequency(docs, top_k_df=top_k_df, min_tf=min_tf)

# 3. Main Logic
## 3.1. Examining and loading data
- Examine the data and explain your findings
- Load the data into proper data structures and get it ready for processing.

In [4]:
print("Setting up...")
_setup()
df_raw: pd.DataFrame = pd.read_csv(DATA_PATH)

Setting up...
Installing nltk packages...
Creating directory: ./data


In [5]:
print("Loading data...")
df_raw: pd.DataFrame = pd.read_csv(DATA_PATH)
df_raw["review_text"] = df_raw["review_text"].fillna("")

Loading data...


In [6]:
print(f"Shape: {df_raw.shape}")

Shape: (61284, 15)


In [7]:
df_raw.head(2)

,product_id,brand_name,review_id,review_title,review_text,author,review_date,review_rating,is_a_buyer,product_title,price,avg_product_rating,product_rating_count,product_tags,product_url
0,781070,Olay,16752142,Worth buying 50g one,Works as it claims. Could see the difference f...,Ashton Dsouza,23/01/2021 15:17,5.0,True,Olay Ultra Lightweight Moisturiser: Luminous W...,1599,4.1,43,NaN,https://www.nykaa.com/olay-ultra-lightweight-m...
1,781070,Olay,14682550,Best cream to start ur day,It does what it claims . Best thing is it smoo...,Amrit Neelam,07/09/2020 15:30,5.0,True,Olay Ultra Lightweight Moisturiser: Luminous W...,1599,4.1,43,NaN,https://www.nykaa.com/olay-ultra-lightweight-m...


## 3.2. Pre-processing data

**Stopwords file (stopwords_en_enhancement.txt)**

We load the enhanced list instead of the baseline stopwords_en.txt because it was audited for this dataset and downstream tasks (especially buyer/sentiment-related signal in Task 3). The original NLTK-style file had 571 lines / 570 unique tokens (one duplicate: would) and included 26 single-letter entries—those are redundant here because our tokenizer already drops single-character tokens in Step 4.

The audit removed 84 tokens that are often kept in generic stopword lists but that carry evaluative or domain meaning in beauty reviews (e.g. best, better, good, like, love, look/looks/looking, really, well), so stripping them would weaken sentiment and purchase-intent signal for classification. It added 19 informal / conversational tokens (lol, tbh, omg, kinda, …) that add little lexical value but appear often in user-generated text. The final list has 505 entries. In short: ENHANCED_PATH = fewer generic stopwords that hurt our labels, plus chat-style noise words, aligned with the investigation report below.

In [8]:
print("Preprocessing data...")
stopwords = load_stopwords(ENHANCED_PATH)  # or STOPWORDS_PATH
reviews = df_raw["review_text"].tolist()
processed_docs = preprocess(
    reviews[:100], # Get first 100 reviews
    stopwords,
    min_tf=1,   # no tf filtering on tiny corpus
    n_jobs=5,
)

Preprocessing data...


Core 2:   0%|          | 0/20 [00:00<?, ?it/s]

Core 5:   0%|          | 0/20 [00:00<?, ?it/s]

Core 3:   0%|          | 0/20 [00:00<?, ?it/s]

Core 4:   0%|          | 0/20 [00:00<?, ?it/s]

Core 1:   0%|          | 0/20 [00:00<?, ?it/s]

In [ ]:
print("Saving vocabulary...")
save_vocab(processed_docs, VOCAB_PATH)

# 4. Appendix

## ClaudeTextExpert Class

In [47]:
class ClaudeTextExpert:
    """Manage stopword-list auditing via the Claude API."""
 
    _SAMPLE_REVIEWS   = 300   # reviews to sample for prompt context
    _SAMPLE_TOP_TOKENS = 60   # top-N tokens to surface in the prompt
 
    # ── Construction ──────────────────────────────────────────────────────────
 
    def __init__(self, model: str = "claude-sonnet-4-6", max_tokens: int = 2048):
        import anthropic
        self.model      = model
        self.max_tokens = max_tokens
        self.client     = anthropic.Anthropic()   # reads ANTHROPIC_API_KEY from env
 
    # ── Public helpers ────────────────────────────────────────────────────────
 
    @staticmethod
    def show_api_key() -> None:
        key = os.getenv("ANTHROPIC_API_KEY", "")
        if not key:
            print("[API] ANTHROPIC_API_KEY not set.")
            return
        print(f"[API] Key starts with: {key[:8]}…")
 
    # ── Sampling (classmethod — no API needed) ────────────────────────────────
 
    @classmethod
    def get_sample_reviews(
        cls,
        reviews: "pd.DataFrame | list[str]",
        n_reviews: int  = _SAMPLE_REVIEWS,
        top_n: int      = _SAMPLE_TOP_TOKENS,
        seed: int       = 42,
    ) -> dict | None:
        """
        Build the context payload that will be injected into the Claude prompt.
 
        Parameters
        ----------
        reviews : pd.DataFrame or list[str]
            Post-Step-4 data.  Either:
              • DataFrame with a 'review_text' column of space-separated token strings, OR
              • list[str] where each element is one post-Step-4 token stream.
            The caller is responsible for having applied Steps 1-4
            (tokenize → lowercase → remove len<2) before passing data in.
        n_reviews : int   — rows to sample (default 300)
        top_n     : int   — most-frequent tokens to report (default 60)
        seed      : int   — random seed for reproducibility
        """
        # Normalise to flat list[str]
        if isinstance(reviews, pd.DataFrame):
            if "review_text" not in reviews.columns:
                raise ValueError("DataFrame must contain a 'review_text' column.")
            streams_all: list[str] = reviews["review_text"].dropna().astype(str).tolist()
        elif isinstance(reviews, list):
            streams_all = [str(r) for r in reviews if r]
        else:
            raise TypeError(f"reviews must be a DataFrame or list[str], got {type(reviews)}")
 
        total = len(streams_all)
        if total == 0:
            print("[Sampler] No reviews provided — skipping.")
            return None
 
        import random
        random.seed(seed)
        k = min(n_reviews, total)
        sampled = random.sample(streams_all, k)
 
        corpus_counter: Counter = Counter()
        for stream in sampled:
            corpus_counter.update(stream.split())
 
        print(f"[Sampler] {total:,} reviews received — sampled {k} for Claude context.")
        return {
            "sampled_streams": sampled,
            "top_tokens"     : corpus_counter.most_common(top_n),
            "total_reviews"  : total,
            "n_sampled"      : k,
        }
 
    # ── Audit entry-point ─────────────────────────────────────────────────────
 
    def claude_audit(self, report: dict, samples: dict | None = None) -> str:
        """
        Send the stopword list to Claude and return the raw response string.
 
        Parameters
        ----------
        report  : output of StopwordsInvestigator.basic_report()
        samples : output of get_sample_reviews(), or None
        """
        issues_block = self._build_issues_block(report)
        real_data_block = self._build_real_data_block(samples)
        prompt = self._build_prompt(report, issues_block, real_data_block)
 
        print("[Claude] Sending audit request …")
        response = self.client.messages.create(
            model    = self.model,
            max_tokens = self.max_tokens,
            messages = [{"role": "user", "content": prompt}],
        )
        raw = response.content[0].text
 
        # Debug: show raw SUGGESTED_ADDITIONS block so format issues are visible
        m = re.search(r"SUGGESTED_ADDITIONS:\s*(.*?)(?=\n[A-Z_]+:|\Z)", raw, re.DOTALL)
        if m:
            print(f"\n[Debug] Raw SUGGESTED_ADDITIONS block:\n{m.group(1)[:600]}")
        else:
            print("\n[Debug] SUGGESTED_ADDITIONS section not found in response.")
 
        return raw
 
    # ── Response parsing ──────────────────────────────────────────────────────
 
    @staticmethod
    def parse_response(text: str) -> dict:
        """
        Extract SUGGESTED_ADDITIONS and SUMMARY from Claude's reply.
 
        Returns
        -------
        dict with keys:
          audit_issues : list[dict]  — always empty (additions-only; no removals from file)
          suggestions  : list[str]   — clean, validated words
          summary      : str
        """
        def extract_section(label: str) -> str:
            m = re.search(rf"{label}:\s*(.*?)(?=\n[A-Z_]+:|\Z)", text, re.DOTALL)
            return m.group(1).strip() if m else ""
 
        suggest_raw = extract_section("SUGGESTED_ADDITIONS")
        summary_raw = extract_section("SUMMARY")
 
        # Additions-only mode: we do not parse or apply removals from the stopword file.
        audit_issues: list[dict] = []
 
        # ── Parse suggestions — robust multi-format cleaning ─────────────────
        suggestions:      list[str]         = []
        suggestions_skipped: list[tuple]    = []
 
        for line in suggest_raw.splitlines():
            # Strip leading: numbers, dots, dashes, bullets, em-dashes, whitespace
            word = re.sub(r"^[\s\d.\-*•–—]+", "", line).strip().lower()
            # Strip trailing punctuation
            word = word.rstrip(".,;:!? ")
 
            if not word:
                continue
            if len(word) < 2:
                suggestions_skipped.append((word, "len<2"))
                continue
            if not TOKENIZER_PATTERN.fullmatch(word):
                suggestions_skipped.append((word, f"regex mismatch"))
                continue
            suggestions.append(word)
 
        if suggestions_skipped:
            print(f"[Parser]  {len(suggestions_skipped)} words skipped:")
            for w, reason in suggestions_skipped:
                print(f"            '{w}' → {reason}")
        print(f"[Parser]  {len(suggestions)} suggestions accepted.")
 
        return {
            "audit_issues": audit_issues,
            "suggestions" : suggestions,
            "summary"     : summary_raw,
        }
 
    # ── Build enhanced list ───────────────────────────────────────────────────
 
    @staticmethod
    def build_enhanced_list(report: dict, parsed: dict) -> list[str]:
        """
        Produce the final enhanced stopword list:
          • Preserve every unique word from the source file (no filtering by length or regex)
          • Union Claude's validated suggestions (regex + length checks apply only to new words)
 
        Use when the same stopword file is shared across pipelines that may differ on token rules.
 
        Parameters
        ----------
        report : output of StopwordsInvestigator.basic_report()
        parsed : output of parse_response()
        """
        base: set[str] = set(report["unique_list"])
 
        # Track what actually changes
        actually_added:  list[str] = []
        already_present: list[str] = []
 
        for w in parsed["suggestions"]:
            if w in base:
                already_present.append(w)
            else:
                base.add(w)
                actually_added.append(w)
 
        # Logging
        if already_present:
            print(f"[Builder] {len(already_present)} suggestions already present: {already_present}")
        if actually_added:
            print(f"[Builder] {len(actually_added)} genuinely new words added:  {actually_added}")
        else:
            print("[Builder] No new words added — all suggestions already existed in the list.")
 
        return sorted(base)
 
    # ── Private prompt builders ───────────────────────────────────────────────
    @staticmethod
    def _build_issues_block(report: dict) -> str:
        issues: list[str] = []
        if report["duplicates"]:
            issues.append(f"- Duplicates found: {report['duplicates']}")
        if report.get("invalid_tokens"):
            issues.append(
                f"- Words that won't survive the tokenizer regex: {report['invalid_tokens']}"
            )
        if report["single_char"]:
            issues.append(
                f"- Single-char words (Step 4 removes len<2): {report['single_char']}"
            )
        return "\n".join(issues) if issues else "None detected."
 
    @staticmethod
    def _build_real_data_block(samples: dict | None) -> str:
        if not samples:
            return textwrap.dedent("""
                ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
                REAL DATA — not available
                ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
                No post-Step-4 data was provided. Base your suggestions on domain knowledge alone.
            """).strip()
 
        top_token_lines = "\n".join(
            f"  {rank+1:>3}. {tok:<25} (freq: {cnt})"
            for rank, (tok, cnt) in enumerate(samples["top_tokens"])
        )
        stream_lines = "\n".join(
            f"  [{i+1:>3}] {stream}"
            for i, stream in enumerate(samples["sampled_streams"][:20])
        )
        return textwrap.dedent(f"""
            ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
            REAL DATA — REVIEWS AFTER STEPS 1-4 (before stopword removal)
            ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
            Source  : {samples['n_sampled']} randomly sampled reviews out of {samples['total_reviews']:,} total.
            Pipeline: tokenized (regex) → lowercased → len<2 removed.
            Stopwords have NOT been applied yet — this is what the filter sees.
 
            TOP {len(samples['top_tokens'])} MOST FREQUENT TOKENS IN THE SAMPLE
            (Pure filler words appearing here are strong candidates to add as stopwords.)
            {top_token_lines}
 
            SAMPLE TOKEN STREAMS (20 of {samples['n_sampled']} reviews shown)
            {stream_lines}
        """).strip()
 
    @staticmethod
    def _build_prompt(report: dict, issues_block: str, real_data_block: str) -> str:
        return textwrap.dedent(f"""
            You help curate English-oriented stopword lists for NLP and text analytics.

            This SAME list file may be reused across different projects (different tokenizers,
            preprocessing rules, languages/domains, and downstream models). Nothing in this prompt
            assumes one fixed pipeline—treat the current file as the source of truth.

            ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
            WHAT YOU MUST DO (ADDITIONS ONLY)
            ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
            • Propose up to 30 NEW tokens that are ABSENT from the current list and are reasonable
              candidates to drop as stopwords (high-frequency function words, fillers, informal
              spellings of junk words, etc.).
            • Do NOT recommend removing, demoting, merging, replacing, or "cleaning up" any entry
              that already appears in the list. Do not audit or criticize existing lines for
              deletion—the maintainer edits removals manually if needed.
            • Do NOT tailor suggestions to "fix" perceived problems in the list (e.g. single-letter
              rows, duplicates noted below). Those exist for compatibility across pipelines; ignore
              them for removal purposes.

            Each suggested token must match this pattern (same check used when merging suggestions):
              regex r"[a-zA-Z]+(?:[-'][a-zA-Z]+)?" and length ≥ 2.
            One token per line under SUGGESTED_ADDITIONS. Skip words already in the list.

            ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
            GUIDANCE FOR WHAT TO ADD (GENERIC)
            ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
            Prefer tokens that usually carry little lexical meaning for bag-of-words, embeddings,
            or retrieval (articles, pronouns, auxiliaries, conjunctions, fillers, chat shorthand).
            Avoid adding content-heavy words (sentiment, product attributes, named entities) unless
            they are clearly universal fillers in user-generated text.

            ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
            CURRENT STOPWORD LIST ({report['unique_words']} unique words)
            ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
            {', '.join(report['unique_list'])}

            ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
            PRE-FLIGHT ISSUES DETECTED BY LOCAL CHECKS (informational only)
            ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
            {issues_block}

            {real_data_block}

            ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
            YOUR TASK
            ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
            1. SUGGESTED_ADDITIONS — up to 30 missing tokens (rules above).
            2. SUMMARY — one short paragraph explaining why these additions help generic English
               stopword filtering (do not reference one specific assignment notebook or pipeline).

            Reply in EXACTLY this format — no extra sections, no preamble:

            SUGGESTED_ADDITIONS:
            <word>

            SUMMARY:
            <paragraph>
        """).strip()

## Stopword Utilities

In [48]:
def basic_report(words: list[str]) -> dict:
    """Quick sanity checks on the raw file."""
    unique: list[str] = sorted(set(words))
    duplicates: list[str] = [w for w in unique if words.count(w) > 1]
    invalid_tokens: list[str] = [
        w for w in unique
        if not TOKENIZER_PATTERN.fullmatch(w)   # won't survive Step 2 tokenizer
    ]
    single_char: list[str] = [w for w in unique if len(w) < 2]  # will be dropped by Step 4

    return {
        "total_lines"     : len(words),
        "unique_words"    : len(unique),
        "duplicates"      : duplicates,
        "invalid_tokens"  : invalid_tokens,   # won't match the Task-1 regex
        "single_char"     : single_char,       # dropped by Step 4 anyway
        "unique_list"     : unique,
    }


def extract_section(label: str, text: str) -> str:
    pattern = rf"{label}:\s*(.*?)(?=\n[A-Z_]+:|\Z)"
    m = re.search(pattern, text, re.DOTALL)
    return m.group(1).strip() if m else ""


def parse_claude_response(text: str) -> dict:
    """Extract SUGGESTED_ADDITIONS and SUMMARY from Claude's structured reply."""

    suggest_raw = extract_section("SUGGESTED_ADDITIONS", text)
    summary_raw = extract_section("SUMMARY", text)

    audit_issues: list[dict] = []

    # Parse suggested additions (one word per line)
    suggestions = []
    for line in suggest_raw.splitlines():
        word = line.strip().lower().lstrip("•-* ")
        if word and TOKENIZER_PATTERN.fullmatch(word) and len(word) >= 2:
            suggestions.append(word)

    return {
        "audit_issues": audit_issues,
        "suggestions" : suggestions,
        "summary"     : summary_raw,
    }


def build_enhanced_list(
    original: list[str],
    parsed: dict,
    report: dict,
) -> list[str]:
    """
    Preserve every unique word from the loaded stopword file, then add Claude's suggestions.
    The ``original`` argument is kept for API compatibility with the notebook; merging uses
    ``report["unique_list"]`` (same unique set derived from that file).
    New suggestions are still validated to match TOKENIZER_PATTERN and length >= 2.
    """
    base = set(report["unique_list"])

    # Add Claude's suggestions
    for w in parsed["suggestions"]:
        base.add(w)

    return sorted(base)


def save_enhanced(words: list[str], path: str) -> None:
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(words) + "\n")
    print(f"\n[Saved] {len(words)} stopwords → {path}")


def print_report(report: dict, parsed: dict, enhanced: list[str], original: list[str]) -> None:
    print("\n" + "=" * 60)
    print("  STOPWORDS INVESTIGATION REPORT")
    print("=" * 60)

    print(f"\n[Original file]")
    print(f"  Total lines  : {report['total_lines']}")
    print(f"  Unique words : {report['unique_words']}")

    if report["duplicates"]:
        print(f"  Duplicates   : {report['duplicates']}")
    else:
        print(f"  Duplicates   : none")

    if report["invalid_tokens"]:
        print(f"  Invalid (regex won't capture): {report['invalid_tokens']}")

    if report["single_char"]:
        print(f"  Single-char (Step 4 removes): {report['single_char']}")

    print(f"\n[Claude — additions only (existing stopwords unchanged)]")
    print("  Removals from the file are not applied in this workflow.")

    print(f"\n[Claude Suggestions — new additions]")
    if parsed["suggestions"]:
        print("  " + ", ".join(parsed["suggestions"]))
    else:
        print("  None suggested.")

    print(f"\n[Claude Summary]")
    for line in parsed["summary"].splitlines():
        print(f"  {line}")

    added   = set(enhanced) - set(original)
    removed = set(original) - set(enhanced)
    print(f"\n[Enhanced list]")
    print(f"  Words added   : {len(added)}   → {sorted(added) if added else 'none'}")
    print(f"  Words removed : {len(removed)} → {sorted(removed) if removed else 'none'}")
    print(f"  Final count   : {len(enhanced)}")
    print("=" * 60)

In [49]:
from dotenv import load_dotenv
"""
You should have a .env file in the root directory of the project with the following variables:

ANTHROPIC_API_KEY=your_anthropic_api_key
"""
load_dotenv()

True

## Running the Audit

In [50]:
samples = ClaudeTextExpert.get_sample_reviews(df_raw) # Get sample reviews
original_words = load_stopwords(STOPWORDS_PATH) # Load original stopwords
report = basic_report(original_words) # Basic report
print(f"[Loaded] {report['total_lines']} lines, {report['unique_words']} unique words")

[Sampler] 61,284 reviews received — sampled 300 for Claude context.
[Loaded] 571 lines, 570 unique words


In [51]:
expert: ClaudeTextExpert = ClaudeTextExpert()
raw_response: str = expert.claude_audit(report, samples) # Claude audi

[Claude] Sending audit request …

[Debug] Raw SUGGESTED_ADDITIONS block:
amongst
ll
ve
re
couldn
doesn
hadn
hasn
haven
isn
wasn
weren
wouldn
shouldn
didn
there's
it's
i'm
i'd
i'll
we're
they're
you're
he's
she's
what's
that's
who's
let's
don



In [52]:
# 5. Parse
parsed = parse_claude_response(raw_response)

# 6. Build enhanced list
enhanced = build_enhanced_list(original_words, parsed, report)

# 7. Decide whether to save (enhanced = full unique source file ∪ validated suggestions)
baseline_sorted = sorted(report["unique_list"])
needs_update = enhanced != baseline_sorted
if needs_update:
    save_enhanced(enhanced, ENHANCED_PATH)
else:
    print("\n[Info] No changes needed — enhanced list is identical to cleaned original.")
    print(f" {ENHANCED_PATH} was NOT created.")


print_report(report, parsed, enhanced, report["unique_list"])



[Saved] 585 stopwords → ./data/stopwords_en_enhancement.txt

  STOPWORDS INVESTIGATION REPORT

[Original file]
  Total lines  : 571
  Unique words : 570
  Duplicates   : ['would']
  Single-char (Step 4 removes): ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']

[Claude — additions only (existing stopwords unchanged)]
  Removals from the file are not applied in this workflow.

[Claude Suggestions — new additions]
  amongst, ll, ve, re, couldn, doesn, hadn, hasn, haven, isn, wasn, weren, wouldn, shouldn, didn, there's, it's, i'm, i'd, i'll, we're, they're, you're, he's, she's, what's, that's, who's, let's, don

[Claude Summary]
  These additions primarily cover contracted auxiliary and pronoun forms that commonly appear in tokenized user-generated text when an apostrophe is stripped or treated as a token boundary, producing residual fragments like "ll," "ve," "re," and negation stumps like "couldn," "doesn